# ADAUSDT intraminute dynamics

???????? ?????? ????????? minute-level feature dataset ?????????????? ???????? ?????? `intraminute_segments` ? raw `aggTrades`, ?? ??????? raw-????. Minute backbone ??????? ?? raw `klines` ? ???????? ?????????? ?????? ??? ??????? ?????.

In [1]:
import io
import os
import re

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv

BUCKET = "binance-data-downloader"
RAW_PREFIX = "raw"
FEATURES_PREFIX = "features"
INPUT_FEATURE_DATASET_NAME = "intraminute_segments"
FEATURE_DATASET_NAME = "intraminute_dynamics"
SYMBOL = "ADAUSDT"
INTERVAL = "1m"
SKIP_EXISTING = True

FEATURE_COLUMNS = [
    "open_time",
    "L",
    "r_1",
    "r_2",
    "r_3",
    "T",
    "pressure_total",
    "pressure_segment_3",
    "F_concentration",
    "RV",
]
FLOAT_FEATURE_COLUMNS = [c for c in FEATURE_COLUMNS if c != "open_time"]


def make_s3_client():
    load_dotenv()
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("YC_ENDPOINT"),
        region_name=os.getenv("YC_REGION"),
        aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    )

In [2]:
def list_symbol_segment_days(symbol: str, bucket: str = BUCKET, features_prefix: str = FEATURES_PREFIX, interval: str = INTERVAL, s3_client=None) -> list[str]:
    s3 = s3_client or make_s3_client()
    source_prefix = f"{features_prefix.strip('/')}/{INPUT_FEATURE_DATASET_NAME}/symbol={symbol}/interval={interval}/"
    pattern = re.compile(r"/date=(\d{4}-\d{2}-\d{2})/data\.parquet$")
    dates=set()
    paginator=s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            m=pattern.search('/'+obj['Key'])
            if m:
                dates.add(m.group(1))
    return sorted(dates)


def klines_key(symbol: str, date: str, raw_prefix: str = RAW_PREFIX, interval: str = INTERVAL) -> str:
    return f"{raw_prefix.strip('/')}/klines/symbol={symbol}/interval={interval}/date={date}/data.parquet"


def aggtrades_key(symbol: str, date: str, raw_prefix: str = RAW_PREFIX) -> str:
    return f"{raw_prefix.strip('/')}/aggTrades/symbol={symbol}/date={date}/data.parquet"


def segments_key(symbol: str, date: str, features_prefix: str = FEATURES_PREFIX, interval: str = INTERVAL) -> str:
    return f"{features_prefix.strip('/')}/{INPUT_FEATURE_DATASET_NAME}/symbol={symbol}/interval={interval}/date={date}/data.parquet"


def feature_dataset_key(symbol: str, date: str, features_prefix: str = FEATURES_PREFIX, interval: str = INTERVAL) -> str:
    return f"{features_prefix.strip('/')}/{FEATURE_DATASET_NAME}/symbol={symbol}/interval={interval}/date={date}/data.parquet"


def s3_key_exists(s3_client, bucket: str, key: str) -> bool:
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except Exception as exc:
        error_code = getattr(exc, "response", {}).get("Error", {}).get("Code")
        if error_code in {"404", "NoSuchKey", "NotFound"}:
            return False
        raise

In [3]:
def read_minute_backbone_day(symbol: str, date: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, interval: str = INTERVAL, s3_client=None) -> pd.DataFrame:
    s3=s3_client or make_s3_client()
    obj=s3.get_object(Bucket=bucket, Key=klines_key(symbol,date,raw_prefix,interval))
    df=pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=["open_time"])
    if df.empty:
        return pd.DataFrame(columns=["open_time"])
    out=df[["open_time"]].copy()
    out["open_time"]=pd.to_numeric(out["open_time"], errors="coerce").astype("Int64")
    out=out.dropna(subset=["open_time"]).drop_duplicates(subset=["open_time"]).sort_values("open_time").reset_index(drop=True)
    ts=pd.to_datetime(out["open_time"].astype("int64"), unit="ms", utc=True)
    if not ts.dt.second.eq(0).all() or not ts.dt.microsecond.eq(0).all():
        raise ValueError("Minute backbone open_time must be minute-aligned UTC")
    return out


def read_segments_day(symbol: str, date: str, bucket: str = BUCKET, features_prefix: str = FEATURES_PREFIX, interval: str = INTERVAL, s3_client=None) -> pd.DataFrame:
    s3=s3_client or make_s3_client()
    obj=s3.get_object(Bucket=bucket, Key=segments_key(symbol,date,features_prefix,interval))
    required=["open_time","VWAP_segment_1","VWAP_segment_2","VWAP_segment_3","delta_segment_1","delta_segment_2","delta_segment_3","pressure_segment_1","pressure_segment_2","pressure_segment_3"]
    df=pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=required)
    missing=sorted(set(required)-set(df.columns))
    if missing:
        raise ValueError(f"Missing required segment columns: {missing}")
    return df


def read_symbol_aggtrades_day(symbol: str, date: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, s3_client=None) -> pd.DataFrame:
    s3=s3_client or make_s3_client()
    obj=s3.get_object(Bucket=bucket, Key=aggtrades_key(symbol,date,raw_prefix))
    df=pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=["transact_time","price"])
    if df.empty:
        return pd.DataFrame(columns=["transact_time","price"])
    df=df.copy()
    df["transact_time"]=pd.to_numeric(df["transact_time"], errors="coerce").astype("Int64")
    df["price"]=pd.to_numeric(df["price"], errors="coerce")
    return df.dropna(subset=["transact_time","price"]).reset_index(drop=True)

In [4]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    result=np.divide(numerator.astype("float64"), denominator.astype("float64"), out=np.zeros(len(numerator), dtype="float64"), where=denominator.astype("float64").to_numpy()!=0)
    return pd.Series(result, index=numerator.index)


def safe_log_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    valid=(numerator.astype("float64")>0) & (denominator.astype("float64")>0)
    result=np.zeros(len(numerator), dtype="float64")
    result[valid.to_numpy()] = np.log(numerator[valid].astype("float64") / denominator[valid].astype("float64"))
    return pd.Series(result, index=numerator.index)


def build_realized_volatility_for_day(trades: pd.DataFrame) -> pd.DataFrame:
    if trades.empty:
        return pd.DataFrame(columns=["open_time","RV"])
    trades=trades.copy()
    trades["open_time"]=(trades["transact_time"].astype("int64") // 60000) * 60000
    trades=trades.sort_values(["open_time","transact_time"]).reset_index(drop=True)
    trades["price_diff_sq"] = trades.groupby("open_time")["price"].diff().pow(2).fillna(0)
    return trades.groupby("open_time", as_index=False)["price_diff_sq"].sum().rename(columns={"price_diff_sq":"RV"})


def build_intraminute_dynamics_for_day(symbol: str, date: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, features_prefix: str = FEATURES_PREFIX, interval: str = INTERVAL, s3_client=None) -> pd.DataFrame:
    backbone=read_minute_backbone_day(symbol,date,bucket,raw_prefix,interval,s3_client)
    segments=read_segments_day(symbol,date,bucket,features_prefix,interval,s3_client)
    trades=read_symbol_aggtrades_day(symbol,date,bucket,raw_prefix,s3_client)
    if backbone.empty:
        return pd.DataFrame(columns=FEATURE_COLUMNS)
    df=backbone.merge(segments, on="open_time", how="left").fillna(0)
    df["delta_total"] = df["delta_segment_1"] + df["delta_segment_2"] + df["delta_segment_3"]
    df["L"] = safe_divide(df["delta_segment_3"], df["delta_total"])
    df["r_1"] = safe_log_ratio(df["VWAP_segment_2"], df["VWAP_segment_1"])
    df["r_2"] = safe_log_ratio(df["VWAP_segment_3"], df["VWAP_segment_2"])
    df["r_3"] = safe_log_ratio(df["VWAP_segment_3"], df["VWAP_segment_1"])
    df["T"] = safe_divide((df["r_1"] + df["r_2"]).abs(), df["r_1"].abs() + df["r_2"].abs())
    df["pressure_total"] = df["pressure_segment_1"] + df["pressure_segment_2"] + df["pressure_segment_3"]
    df["F_concentration"] = safe_divide(df["pressure_segment_3"], df["pressure_total"])
    rv=build_realized_volatility_for_day(trades)
    df=df.merge(rv, on="open_time", how="left").fillna(0)
    df=df.replace([np.inf,-np.inf], np.nan).fillna(0)
    for c in FLOAT_FEATURE_COLUMNS:
        df[c]=df[c].astype("float32")
    return df[FEATURE_COLUMNS].sort_values("open_time").reset_index(drop=True)

In [5]:
def write_intraminute_dynamics_for_symbol_to_s3(symbol: str = SYMBOL, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, features_prefix: str = FEATURES_PREFIX, interval: str = INTERVAL, skip_existing: bool = SKIP_EXISTING, s3_client=None) -> pd.DataFrame:
    s3=s3_client or make_s3_client()
    dates=list_symbol_segment_days(symbol,bucket,features_prefix,interval,s3)
    if not dates:
        raise FileNotFoundError(f"No intraminute_segments days found for symbol={symbol}")
    rows=[]
    for date in dates:
        key=feature_dataset_key(symbol,date,features_prefix,interval)
        if skip_existing and s3_key_exists(s3,bucket,key):
            print(f"Skip exists: s3://{bucket}/{key}")
            rows.append({"date":date,"rows":None,"key":key,"status":"skipped"})
            continue
        feature_df=build_intraminute_dynamics_for_day(symbol,date,bucket,raw_prefix,features_prefix,interval,s3)
        buffer=io.BytesIO(); feature_df.to_parquet(buffer,index=False,engine="pyarrow",compression="zstd")
        s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
        print(f"Uploaded: s3://{bucket}/{key} rows={len(feature_df)}")
        rows.append({"date":date,"rows":len(feature_df),"key":key,"status":"uploaded"})
    return pd.DataFrame(rows)

In [6]:
# Preflight: ???????????? ????????????? raw aggTrades ? ????????? config.yaml.
from config_loader import load_config
from aggtrades_backfill import backfill_missing_aggtrades_dates

config = load_config("config.yaml")
s3 = make_s3_client()
aggtrades_backfill_results = []

for symbol in config["symbols"]:
    aggtrades_backfill_results.append(
        backfill_missing_aggtrades_dates(
            symbol=symbol,
            start_date=config["date_range"]["start"],
            end_date=config["date_range"]["end"],
            bucket=config["storage"]["bucket"],
            raw_prefix=config["storage"]["prefix"],
            retries=config["download"]["retries"],
            timeout=tuple(config["download"]["timeout"]),
            s3_client=s3,
        )
    )

aggtrades_backfill_results = (
    pd.concat(aggtrades_backfill_results, ignore_index=True)
    if aggtrades_backfill_results
    else pd.DataFrame(columns=["symbol", "date", "rows", "key", "status"])
)

display(aggtrades_backfill_results)
if not aggtrades_backfill_results.empty:
    print(aggtrades_backfill_results["status"].value_counts(dropna=False))
else:
    print("No missing raw aggTrades days found.")

,symbol,date,rows,key,status


No missing raw aggTrades days found.


In [6]:
# ??????? ???????? ?? ????? ??? ????? ???????? ???????.
s3 = make_s3_client()
dates = list_symbol_segment_days(SYMBOL, s3_client=s3)
example_date = dates[1]
example_features = build_intraminute_dynamics_for_day(SYMBOL, example_date, s3_client=s3)
print(example_date, example_features.shape)
display(example_features.head())
display(example_features.tail())
display(example_features.dtypes)

2020-02-02 (1440, 10)


,open_time,L,r_1,r_2,r_3,T,pressure_total,pressure_segment_3,F_concentration,RV
0,1580601600000,0.449493,0.000071,0.000296,0.000366,1.000000,0.249075,0.471116,1.891464,5.800330e-09
1,1580601660000,0.805610,0.000554,-0.000976,-0.000422,0.275940,4.142570,3.569106,0.861568,2.250017e-08
2,1580601720000,-0.151862,-0.000671,-0.000141,-0.000812,1.000000,7.279718,1.161420,0.159542,4.180043e-08
3,1580601780000,-1.187628,0.000064,0.000739,0.000803,1.000000,2.644039,0.975214,0.368835,2.680127e-08
4,1580601840000,1.935530,0.001393,-0.001275,0.000118,0.044132,5.015728,1.305408,0.260263,4.580049e-08


,open_time,L,r_1,r_2,r_3,T,pressure_total,pressure_segment_3,F_concentration,RV
1435,1580687700000,0.867711,-0.000911,0.000105,-0.000805,0.793097,-0.646185,0.931575,-1.441653,2.799782e-09
1436,1580687760000,-0.022241,-0.001191,0.001360,0.000169,0.066258,2.838001,0.808926,0.285034,1.239933e-08
1437,1580687820000,1.099987,0.000000,0.000351,0.000000,1.000000,0.163105,0.081573,0.500123,2.000217e-10
1438,1580687880000,-2.556700,0.000186,-0.000177,0.000008,0.023335,0.138611,0.078331,0.565113,3.998943e-10
1439,1580687940000,1.055868,0.000893,-0.000226,0.000667,0.595576,7.343301,-0.231282,-0.031496,1.420050e-08


open_time               Int64
L                     float32
r_1                   float32
r_2                   float32
r_3                   float32
T                     float32
pressure_total        float32
pressure_segment_3    float32
F_concentration       float32
RV                    float32
dtype: object

In [7]:
# ???????? ?????? ???? ??????? parquet-?????? ????????? ? S3.
# ??????????????, ????? ?????? ????? ????????? ?????? ????????.
# write_results = write_intraminute_dynamics_for_symbol_to_s3(s3_client=s3)
# display(write_results)

In [ ]:
# ?????? ?? ???? ?????? ?? config.yaml.
from config_loader import load_config

config = load_config("config.yaml")
config_symbols = config["symbols"]
config_interval = config["interval"]
config_start_date = pd.Timestamp(config["date_range"]["start"]).date()
config_end_date = pd.Timestamp(config["date_range"]["end"]).date()
config_bucket = config["storage"]["bucket"]
config_raw_prefix = config["storage"]["prefix"]

s3 = make_s3_client()
all_write_results = []

for symbol in config_symbols:
    available_dates = list_symbol_segment_days(
        symbol=symbol,
        bucket=config_bucket,
        features_prefix=FEATURES_PREFIX,
        interval=config_interval,
        s3_client=s3,
    )
    selected_dates = [
        date
        for date in available_dates
        if config_start_date <= pd.Timestamp(date).date() <= config_end_date
    ]

    if not selected_dates:
        print(f"No intraminute_segments days found in config range for symbol={symbol}")
        continue

    rows = []
    for date in selected_dates:
        key = feature_dataset_key(
            symbol=symbol,
            date=date,
            interval=config_interval,
        )

        if SKIP_EXISTING and s3_key_exists(s3, config_bucket, key):
            print(f"Skip exists: s3://{config_bucket}/{key}")
            rows.append({"symbol": symbol, "date": date, "rows": None, "key": key, "status": "skipped"})
            continue

        feature_df = build_intraminute_dynamics_for_day(
            symbol=symbol,
            date=date,
            bucket=config_bucket,
            raw_prefix=config_raw_prefix,
            features_prefix=FEATURES_PREFIX,
            interval=config_interval,
            s3_client=s3,
        )

        buffer = io.BytesIO()
        feature_df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
        s3.put_object(Bucket=config_bucket, Key=key, Body=buffer.getvalue())

        print(f"Uploaded: s3://{config_bucket}/{key} rows={len(feature_df)}")
        rows.append({"symbol": symbol, "date": date, "rows": len(feature_df), "key": key, "status": "uploaded"})

    all_write_results.append(pd.DataFrame(rows))

write_results_config_period = (
    pd.concat(all_write_results, ignore_index=True)
    if all_write_results
    else pd.DataFrame(columns=["symbol", "date", "rows", "key", "status"])
)

display(write_results_config_period)
print(write_results_config_period["status"].value_counts(dropna=False))